# GEAK Agent: HIP Kernel Optimization

**GEAK** is an AI-powered system that automatically optimizes HIP GPU kernels. It uses Large Language Models (LLMs) like Claude or GPT to analyze naive kernel implementations and generate optimized versions through an evolutionary approach.

### What This Notebook Does
1. **Sets up** the GEAK agent environment and API connections
2. **Selects** which GPU kernels to optimize (GEMM, Softmax, LayerNorm, etc.)
3. **Runs** the AI optimization loop that iteratively improves kernel performance
4. **Verifies** correctness by comparing outputs against baseline implementations
5. **Analyzes** speedup results with performance metrics and visualizations

### Prerequisites
- AMD GPU with ROCm/HIP support
- API key for Claude (Anthropic) or OpenAI
- Python environment with required dependencies

## Step 1: Setup Repository

In this step, we configure the environment variables and clone the GEAK agent repository.

**What happens here:**
- **`.env` file creation**: Stores your API keys securely. You must replace `your-claude-key-here` or `your-openai-key-here` with your actual API keys.
- **Repository cloning**: Downloads the GEAK-agent codebase from GitHub and checks out the `neurips` branch.

**Configuration Options:**
| Parameter | Description |
|-----------|-------------|
| `LLM_PROVIDER` | Choose `claude` or `openai` as your LLM backend |
| `TEMPERATURE` | Controls randomness in LLM responses (0.0 = deterministic, 1.0 = creative) |
| `MAX_ITERATION` | Number of optimization iterations per kernel |
| `DESCENDANT_NUM` | Number of candidate variants generated per iteration |
| `ANCESTOR_NUM` | Number of top performers kept for next iteration |

In [ ]:
# Edit .env file with your API keys
env_content = """# GEAK Agent API Keys
CLAUDE_API_KEY=your-claude-key-here
OPENAI_API_KEY=your-openai-key-here
# LLM Provider: "claude" or "openai"
LLM_PROVIDER=claude

# Model Selection
CLAUDE_MODEL=claude-sonnet-4-5
OPENAI_MODEL=gpt-5-mini

# Generation Parameters
TEMPERATURE=0.8
MAX_TOKENS=4096

# Optimization Settings
MAX_ITERATION=3
DESCENDANT_NUM=3
ANCESTOR_NUM=4
"""

with open('.env', 'w') as f:
    f.write(env_content)
    
print("✅ .env file updated!")
!cat .env

In [ ]:
import os
import subprocess
import sys
import json
import yaml
from dotenv import load_dotenv

workspace_dir = os.path.join(os.getcwd())
os.chdir(workspace_dir)

repo_dir = "GEAK-agent"
if not os.path.exists(repo_dir):
    print("Cloning GEAK repository...")
    subprocess.run(["git", "clone", "https://github.com/AMD-AGI/GEAK-agent.git"], check=False)
    subprocess.run(["git", "-C", repo_dir, "checkout", "neurips"], check=True)
else:
    print("✓ Repository exists")

print("✓ Setup complete")

## Step 2: Configure LLM

This step loads the API configuration from `.env` and creates a YAML config file that the GEAK agent will use.

**What happens here:**
1. **Loads environment variables** from the `.env` file you created in Step 1
2. **Determines the LLM provider** (Claude or OpenAI) and sets appropriate API endpoints
3. **Writes the GEAK config** to `hipbench_gaagent_config.yaml` with all optimization parameters

**Supported Models:**
- **Claude**: `claude-sonnet-4-5`, `claude-3-5-sonnet-20241022`, etc.
- **OpenAI**: `gpt-4-turbo`, `gpt-4o`, etc.

The config file tells GEAK where to find kernel instructions, how many iterations to run, and where to save results.

In [ ]:
load_dotenv(".env")

CLAUDE_API_KEY = os.getenv('CLAUDE_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
LLM_PROVIDER = os.getenv('LLM_PROVIDER', 'claude').lower()
CLAUDE_MODEL = os.getenv('CLAUDE_MODEL', 'claude-3-5-sonnet-20241022')
OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4-turbo')

MAX_ITERATION = int(os.getenv('MAX_ITERATION', '5'))
DESCENDANT_NUM = int(os.getenv('DESCENDANT_NUM', '2'))
ANCESTOR_NUM = int(os.getenv('ANCESTOR_NUM', '3'))

if LLM_PROVIDER == "claude":
    API_KEY = CLAUDE_API_KEY
    API_BASE_URL = "https://api.anthropic.com/v1"
    MODEL_ID = CLAUDE_MODEL
elif LLM_PROVIDER == "openai":
    API_KEY = OPENAI_API_KEY
    API_BASE_URL = "https://api.openai.com/v1"
    MODEL_ID = OPENAI_MODEL

print(f"Provider: {LLM_PROVIDER}")
print(f"Model: {MODEL_ID}")
print(f"API Key: {'SET' if API_KEY else 'NOT SET'}")
print(f"Max Iterations: {MAX_ITERATION}")

config_file = "GEAK-agent/src/configs/hipbench_gaagent_config.yaml"
config = {
    'api_key': API_KEY,
    'model_id': MODEL_ID,
    'temperature': 1.0,
    'api_base_url': API_BASE_URL,
    'max_tokens': 4096,
    'instruction_path': 'FromRe_instructions.json',
    'corpus_path': '../example_hip',
    'max_iteration': MAX_ITERATION,
    'descendant_num': DESCENDANT_NUM,
    'ancestor_num': ANCESTOR_NUM,
    'result_path': None,
    'output_path': f'{workspace_dir}/iteration_logs/iter',
    'multi_thread': False
}
os.makedirs(os.path.dirname(config_file), exist_ok=True)
with open(config_file, 'w') as f:
    yaml.dump(config, f)

print("✓ Config updated")

In [ ]:
# Cell 4: Verify Public API Support

import os

public_model_path = "GEAK-agent/src/models/PublicLLM.py"
main_script_path = "GEAK-agent/src/main_gaagent_hip_public.py"

if os.path.exists(public_model_path) and os.path.exists(main_script_path):
    print("✓ PublicLLMModel found!")
    print("✓ Public API script found!")
else:
    print("✗ Public API files not found. They should have been created.")
    print("\nMake sure you have:")
    print("  - GEAK-agent/src/models/PublicLLM.py")
    print("  - GEAK-agent/src/main_gaagent_hip_public.py")

## Step 3: Test LLM Connection

Before running the full optimization pipeline, we verify that the LLM API connection is working properly.

**What happens here:**
1. **Imports the PublicLLMModel** wrapper from the GEAK codebase
2. **Creates a model instance** with your API credentials
3. **Sends a simple test prompt** ("Say 'OK' only") to verify connectivity

**Troubleshooting:**
- ❌ **"Connection failed"**: Check that your API key is valid and has available credits
- ❌ **"Module not found"**: Ensure Step 1 completed successfully and the repo was cloned
- ✅ **"Connection successful"**: You're ready to proceed with kernel optimization!

In [ ]:
sys.path.insert(0, "GEAK-agent/src")

from models.PublicLLM import PublicLLMModel

model = PublicLLMModel(
    api_key=API_KEY,
    model_id=MODEL_ID
)

print(f"Provider: {model.provider}")
print(f"Model: {model.model_id}")

try:
    response = model.generate([{"role": "user", "content": "Say 'OK' only."}])
    print(f"✓ Connection successful")
    print(f"Response: {response}")
except Exception as e:
    print(f"✗ Connection failed: {e}")

In [ ]:
sys.path.insert(0, "GEAK-agent/src")

from models.PublicLLM import PublicLLMModel

model = PublicLLMModel(
    api_key=API_KEY,
    model_id=MODEL_ID
)

print(f"Provider: {model.provider}")
print(f"Model: {model.model_id}")

try:
    response = model.generate([{"role": "user", "content": "Say 'OK' only."}])
    print(f"✓ Connection successful")
    print(f"Response: {response}")
except Exception as e:
    print(f"✗ Connection failed: {e}")

## Step 4: Select Kernels

Choose which GPU kernels you want GEAK to optimize. Each kernel has a naive baseline implementation that the AI will attempt to improve.

**Available Kernels:**

| Kernel | Description | Key Optimizations |
|--------|-------------|-------------------|
| `silu` | SiLU activation (x * sigmoid(x)) | Vectorized bf16 loads, larger block size |
| `gemm_naive` | General Matrix Multiplication | Shared memory tiling, thread-level tiling |
| `softmax_naive` | Softmax normalization | Parallel reduction, vectorized loads |
| `conv_naive` | 2D Convolution | Shared memory for tiles and weights |
| `attention_naive` | Self-attention mechanism | 2D blocks, fused softmax |
| `transpose_naive` | Matrix Transpose | 2D blocks, shared memory for coalescing |

**How to Select:**
Set `True` for kernels you want to optimize, `False` to skip them. Start with one kernel to test the pipeline before running multiple.

In [ ]:
ENABLE_KERNELS = {
    "silu": True,  
    "gemm_naive": True,
    "softmax_naive": True,
    "conv_naive": True,
    "attention_naive": True,
    "transpose_naive": True,
}

# Master kernel configurations - these are NEVER modified
# FromRe_instructions.json is only written with selected kernels for GEAK to process
MASTER_KERNELS = {
    "gemm_naive": {
        "task": "gemm_naive",
        "instruction": "Optimize the naive GEMM kernel. Current version uses small 8x8 thread blocks, no shared memory, no tiling. Key optimizations: use shared memory for A and B tiles, increase block size, use thread-level tiling (each thread computes multiple elements), loop unrolling. MUST keep kernel name: gemm_naive_kernel and signature: (float* C, const float* A, const float* B, int M, int N, int K).",
        "file_path": "gemm_naive",
        "file_name": "gemm_naive.hip"
    },
    "softmax_naive": {
        "task": "softmax_naive",
        "instruction": "Optimize the naive softmax kernel. Current version uses only 32 threads per block with sequential reduction in thread 0. Key optimizations: increase block size to 256, use warp shuffle for parallel reduction, use vectorized loads. MUST keep kernel name: softmax_naive_kernel and signature: (float* output, const float* input, int batch_size, int feature_dim).",
        "file_path": "softmax_naive",
        "file_name": "softmax_naive.hip"
    },
    "conv_naive": {
        "task": "conv_naive",
        "instruction": "Optimize the naive 2D convolution kernel. Current version has no shared memory, reads kernel weights from global memory repeatedly. Key optimizations: use shared memory for input tiles and kernel weights, use register blocking. MUST keep kernel name: conv2d_naive_kernel.",
        "file_path": "conv_naive",
        "file_name": "conv_naive.hip"
    },
    "attention_naive": {
        "task": "attention_naive",
        "instruction": "Optimize the naive attention kernel. Current version uses flat indexing with small block size, no shared memory. Key optimizations: use 2D thread blocks, shared memory tiling for Q/K/V, fused softmax computation. MUST keep kernel names: attention_naive_kernel and attention_softmax_kernel.",
        "file_path": "attention_naive",
        "file_name": "attention_naive.hip"
    },
    "silu": {
        "task": "silu",
        "instruction": "Optimize SiLU activation kernel. Current version uses only 32 threads per block which is very inefficient. Key optimizations: increase block size to 256-1024 threads, use vectorized bf16 loads (bf16x2 or bf16x4), use __expf for faster exp, use __forceinline__. MUST keep kernel name: silu_mul_kernel and signature: (bf16* out, const bf16* in, int64_t B, int64_t H).",
        "file_path": "silu",
        "file_name": "silu.hip"
    },
    "transpose_naive": {
        "task": "transpose_naive",
        "instruction": "Optimize the naive transpose kernel. Current version uses flat 1D indexing with small block size, causing non-coalesced memory writes. Key optimizations: use 2D thread blocks (e.g., 32x32), use shared memory tile to enable coalesced reads and writes, handle bank conflicts. MUST keep kernel name: transpose_naive_kernel and signature: (float* output, const float* input, int rows, int cols).",
        "file_path": "transpose_naive",
        "file_name": "transpose_naive.hip"
    }
}

# Select kernels based on ENABLE_KERNELS
selected = [MASTER_KERNELS[k] for k in ENABLE_KERNELS if ENABLE_KERNELS[k] and k in MASTER_KERNELS]

# Write selected kernels to FromRe_instructions.json for GEAK to process
instr_file = "GEAK-agent/example_hip/FromRe_instructions.json"
with open(instr_file, 'w') as f:
    json.dump(selected, f, indent=2)

print(f"✓ Selected {len(selected)} kernel(s)")
for k in selected:
    print(f"  - {k['task']}")


## Step 5: Run GEAK Agent

This is the main optimization loop where GEAK uses the LLM to iteratively improve kernel performance.

**The Evolutionary Optimization Process:**

```
┌─────────────────────────────────────────────────────────────┐
│  1. ANALYZE: LLM reads naive kernel code                    │
│  2. GENERATE: LLM proposes optimized variants (descendants) │
│  3. COMPILE: HIP compiler builds each variant               │
│  4. BENCHMARK: Run on GPU to measure performance            │
│  5. SELECT: Keep best performers (ancestors) for next round │
│  6. REPEAT: Loop for MAX_ITERATION cycles                   │
└─────────────────────────────────────────────────────────────┘
```

**What Gets Saved:**
- `results_{kernel_name}/` — Dedicated folder for each kernel
- `iter_*.hip` — Generated kernel variants from each iteration
- `optimization_output.txt` — Full LLM conversation and optimization logs
- `*_baseline.hip` — Backup of the original naive implementation

**⏱️ Expected Runtime:** 2-10 minutes per kernel depending on iteration count and LLM response time.

In [ ]:
import time
import re
import shutil

python = f"python"

baseline_results = {}
optimization_results = {}
kernel_timing = {}  # Track processing time per kernel

total_start_time = time.time()

for kernel in selected:
    kernel_name = kernel['task']
    kernel_start_time = time.time()
    
    print(f"\n{'='*60}")
    print(f"Optimizing: {kernel_name}")
    print(f"{'='*60}")
    
    # Create dedicated results folder for this kernel
    kernel_results_dir = os.path.join(workspace_dir, f"results_{kernel_name}")
    os.makedirs(kernel_results_dir, exist_ok=True)
    
    # Update config for this specific kernel
    config_file = "GEAK-agent/src/configs/hipbench_gaagent_config.yaml"
    kernel_output_path = os.path.join(kernel_results_dir, "iter")
    
    # Create instruction file with only this kernel
    kernel_instruction_file = os.path.join(kernel_results_dir, "instruction.json")
    with open(kernel_instruction_file, 'w') as f:
        json.dump([kernel], f, indent=2)
    
    # Update config for this kernel
    config = {
        'api_key': API_KEY,
        'model_id': MODEL_ID,
        'temperature': 0.8,
        'api_base_url': API_BASE_URL,
        'max_tokens': 4096,
        'instruction_path': kernel_instruction_file,
        'corpus_path': '../example_hip',
        'max_iteration': MAX_ITERATION,
        'descendant_num': DESCENDANT_NUM,
        'ancestor_num': ANCESTOR_NUM,
        'result_path': None,
        'output_path': kernel_output_path,
        'multi_thread': False
    }
    
    with open(config_file, 'w') as f:
        yaml.dump(config, f)
    
    print(f"Results will be saved to: {kernel_results_dir}")
    
    # Measure baseline performance if a primary file is defined
    file_name = kernel.get('file_name')
    baseline_backup = None
    if file_name:
        print(f"Measuring baseline for {kernel_name}...")
        baseline_file = f"example_hip/{kernel['file_path']}/{file_name}"
        baseline_abs = os.path.join("GEAK-agent", baseline_file)
        baseline_backup = os.path.join(kernel_results_dir, f"{file_name.replace('.hip', '')}_baseline.hip")
        shutil.copyfile(baseline_abs, baseline_backup)
        print(f"  Baseline copy saved to: {baseline_backup}")
    else:
        print("  (No primary HIP file defined for baseline copy)")
    
    baseline_results[kernel_name] = {
        'measured': False,
        'perf': None,
        'correctness': False
    }
    
    # Run GEAK agent optimization
    optimization_start = time.time()
    os.chdir(os.path.join(workspace_dir, "GEAK-agent", "src"))
    result = subprocess.run([python, "main_gaagent_hip_public.py"], 
                          capture_output=True, text=True, timeout=1200)
    os.chdir(workspace_dir)
    optimization_duration = time.time() - optimization_start
    
    kernel_end_time = time.time()
    kernel_duration = kernel_end_time - kernel_start_time
    
    # Store timing information
    kernel_timing[kernel_name] = {
        'total_time': kernel_duration,
        'optimization_time': optimization_duration,
        'setup_time': kernel_duration - optimization_duration
    }
    
    optimization_results[kernel_name] = {
        'completed': result.returncode == 0,
        'output': result.stdout,
        'errors': result.stderr,
        'results_dir': kernel_results_dir,
        'processing_time': kernel_duration,
        'optimization_time': optimization_duration
    }
    
    # Save detailed output to results folder
    with open(os.path.join(kernel_results_dir, "optimization_output.txt"), 'w') as f:
        f.write(result.stdout)
    with open(os.path.join(kernel_results_dir, "optimization_errors.txt"), 'w') as f:
        f.write(result.stderr)
    
    # Save timing info to results folder
    with open(os.path.join(kernel_results_dir, "timing.txt"), 'w') as f:
        f.write(f"Kernel: {kernel_name}\n")
        f.write(f"Total Processing Time: {kernel_duration:.2f} seconds ({kernel_duration/60:.2f} minutes)\n")
        f.write(f"Optimization Time: {optimization_duration:.2f} seconds ({optimization_duration/60:.2f} minutes)\n")
        f.write(f"Setup Time: {kernel_duration - optimization_duration:.2f} seconds\n")
    
    if result.returncode == 0:
        print(f"✓ {kernel_name} optimization complete")
        print(f"   Results saved to: {kernel_results_dir}")
    else:
        print(f"✗ {kernel_name} optimization failed")
        print(f"   Error: {result.stderr[:500]}")
    
    # Report timing for this kernel
    print(f"\n⏱️  {kernel_name} Processing Time:")
    print(f"   Total: {kernel_duration:.2f}s ({kernel_duration/60:.2f} min)")
    print(f"   Optimization: {optimization_duration:.2f}s ({optimization_duration/60:.2f} min)")
    print(f"   Setup: {kernel_duration - optimization_duration:.2f}s")
    
    # Restore baseline kernel back to GEAK-agent folder
    if baseline_backup and os.path.exists(baseline_backup):
        baseline_abs = os.path.join(workspace_dir, "GEAK-agent", "example_hip", kernel['file_path'], file_name)
        shutil.copyfile(baseline_backup, baseline_abs)
        print(f"\n🔄 Restored baseline kernel: {baseline_abs}")

total_duration = time.time() - total_start_time

print("\n" + "="*60)
print("All kernels processed")
print("="*60)

# Print timing summary
print("\n" + "="*60)
print("⏱️  TIMING SUMMARY")
print("="*60)
print(f"{'Kernel':<25} {'Total (s)':<12} {'Total (min)':<12} {'Optimization (s)':<18}")
print("-"*60)
for kernel_name, timing in kernel_timing.items():
    print(f"{kernel_name:<25} {timing['total_time']:<12.2f} {timing['total_time']/60:<12.2f} {timing['optimization_time']:<18.2f}")
print("-"*60)
print(f"{'TOTAL':<25} {total_duration:<12.2f} {total_duration/60:<12.2f}")
print("="*60)

## Step 6: Verify Baseline vs Optimized Outputs

**Correctness is Critical!** A faster kernel is useless if it produces wrong results. This step validates that optimized kernels match the baseline output.

**Verification Process:**
1. **Run baseline**: Execute the naive kernel and dump output to `baseline_output.bin`
2. **Run each iteration**: Execute optimized variants with identical inputs
3. **Compare outputs**: Byte-for-byte comparison against baseline
4. **Measure performance**: Record execution time for each variant

**Output Status:**
- ✅ **MATCH**: Optimized kernel produces identical output — safe to use!
- ❌ **MISMATCH**: Output differs from baseline — kernel has a bug
- ⚠️ **FAIL**: Kernel crashed or failed to compile

**Why This Matters:**
LLMs can generate code that compiles and runs but produces incorrect results due to:
- Off-by-one indexing errors
- Race conditions in parallel code
- Incorrect shared memory synchronization
- Numerical precision issues


In [ ]:
import glob
import shutil
import re

print("\n" + "="*80)
print("RUNNING BASELINE VS OPTIMIZED COMPARISON")
print("="*80)

comparison_report = []

def extract_perf(text):
    if not text:
        return None
    # Check for ms format first
    match = re.search(r"Perf:\s*([0-9.]+)\s*ms", text)
    if match:
        return float(match.group(1))
    match = re.search(r"Performance:\s*([0-9.]+)\s*ms", text)
    if match:
        return float(match.group(1))
    # Check for us (microseconds) format - convert to ms
    match = re.search(r"Perf:\s*([0-9.]+)\s*us", text)
    if match:
        return float(match.group(1)) / 1000.0  # Convert us to ms
    return None

for kernel in selected:
    kernel_name = kernel["task"]
    file_name = kernel.get("file_name")
    
    # Skip kernels without a primary file (e.g., mmcv with multiple sub-kernels)
    if not file_name:
        print(f"\n{'-'*60}\nKernel: {kernel_name}")
        print("  (No primary HIP file defined; skipping baseline comparison)")
        comparison_report.append({"kernel": kernel_name, "baseline": False, "baseline_perf": None, "optimized": []})
        continue
    
    kernel_dir = os.path.join(workspace_dir, "GEAK-agent","example_hip", kernel["file_path"])
    hip_target = os.path.join(kernel_dir, file_name)
    kernel_results_dir = os.path.join(workspace_dir, f"results_{kernel_name}")
    baseline_backup = os.path.join(kernel_results_dir, f"{file_name.replace('.hip', '')}_baseline.hip")
    baseline_dump = os.path.join(kernel_results_dir, "baseline_output.bin")
    iter_files = sorted(glob.glob(os.path.join(kernel_results_dir, "iter_*.hip")))
    test_binary = f"./test_{kernel_name.replace('_naive', '')}"

    if not os.path.exists(baseline_backup):
        print(f"✗ Missing baseline backup for {kernel_name}, skipping")
        comparison_report.append({"kernel": kernel_name, "baseline": False, "baseline_perf": None, "optimized": []})
        continue

    print(f"\n{'-'*60}\nKernel: {kernel_name}")

    def run_test_binary(extra_env):
        env = os.environ.copy()
        env.update(extra_env)
        try:
            subprocess.run(["make", "clean"], cwd=kernel_dir, check=True, capture_output=True)
            subprocess.run(["make"], cwd=kernel_dir, check=True, capture_output=True)
            result = subprocess.run([test_binary], cwd=kernel_dir, env=env, capture_output=True, text=True)
            perf = extract_perf(result.stdout)
            return result, perf
        except subprocess.CalledProcessError as err:
            print(f"  ✗ Build/Test failed: {err}")
            return None

    shutil.copyfile(baseline_backup, hip_target)
    baseline_tuple = run_test_binary({"GEAK_DUMP_OUTPUT": baseline_dump})
    baseline_result, baseline_perf = (baseline_tuple if baseline_tuple else (None, None))
    baseline_pass = baseline_result is not None and baseline_result.returncode == 0
    perf_msg = f" | Perf: {baseline_perf:.4f} ms" if baseline_pass and baseline_perf else ""
    print(f"  Baseline run: {'PASS' if baseline_pass else 'FAIL'}{perf_msg}")

    iter_results = []
    if baseline_pass:
        for iter_file in iter_files:
            iter_name = os.path.basename(iter_file)
            shutil.copyfile(iter_file, hip_target)
            iter_output_dump = os.path.join(kernel_results_dir, f"{iter_name}.bin")
            env_vars = {
                "GEAK_COMPARE_WITH": baseline_dump,
                "GEAK_DUMP_OUTPUT": iter_output_dump
            }
            iter_tuple = run_test_binary(env_vars)
            iter_result, iter_perf = (iter_tuple if iter_tuple else (None, None))
            iter_pass = iter_result is not None and iter_result.returncode == 0
            status = "MATCH" if iter_pass else "MISMATCH"
            perf_msg = f" | Perf: {iter_perf:.4f} ms" if iter_pass and iter_perf else ""
            print(f"    {iter_name}: {status}{perf_msg}")
            if iter_result is not None and iter_result.stdout:
                print("      output:")
                print("      " + iter_result.stdout.strip().replace("\n", "\n      "))
            iter_results.append({"iteration": iter_name, "match": iter_pass, "perf": iter_perf})
        # CRITICAL: Always restore baseline after comparison to keep GEAK-agent in default state
        shutil.copyfile(baseline_backup, hip_target)
        print(f"  🔄 Restored baseline: {hip_target}")
    else:
        print("  Skipping optimized comparisons due to baseline failure")

    comparison_report.append({
        "kernel": kernel_name,
        "baseline": baseline_pass,
        "baseline_perf": baseline_perf,
        "optimized": iter_results
    })

print("\n" + "="*80)
print("COMPARISON SUMMARY")
print("="*80)
for entry in comparison_report:
    print(f"Kernel: {entry['kernel']}")
    print(f"  Baseline run: {'PASS' if entry['baseline'] else 'FAIL'}")
    for item in entry["optimized"]:
        status = "MATCH" if item["match"] else "MISMATCH"
        perf_msg = f" ({item['perf']:.4f} ms)" if item.get('perf') else ""
        print(f"    {item['iteration']}: {status}{perf_msg}")
    if not entry["optimized"]:
        print("    No optimized iterations evaluated")



## Step 7: Analyze Results

Final analysis that aggregates all optimization results into a comprehensive summary.

**What You'll See:**

1. **Per-Kernel Results:**
   - Compilation status (did the optimized code compile?)
   - Execution status (did it run without crashing?)
   - Baseline vs optimized performance in milliseconds
   - Speedup factor (e.g., 2.5x faster)

2. **Summary Table:**
   - Side-by-side comparison of all kernels
   - Quick ✓/✗ indicators for improvement

3. **Visualizations:**
   - Bar charts comparing baseline vs optimized latency
   - Speedup annotations on each comparison

**Generated Files:**
- `summary.txt` — Text summary in each kernel's results folder
- `baseline_vs_optimized.png` — Performance comparison chart

**Interpreting Speedup:**
- **> 1.0x**: Optimization succeeded (faster than baseline)
- **= 1.0x**: No improvement
- **< 1.0x**: Regression (slower than baseline — investigate why!)

In [ ]:
import glob
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import os 

workspace_dir = os.path.join(os.getcwd())
os.chdir(workspace_dir)

print("\n" + "="*80)
print("OPTIMIZATION RESULTS SUMMARY")
print("="*80)

comparison_map = {entry['kernel']: entry for entry in comparison_report} if 'comparison_report' in globals() else {}

results_table = []
plot_data = []

for kernel in selected:
    kernel_name = kernel['task']
    print(f"\n{'='*60}")
    print(f"Results for: {kernel_name}")
    print(f"{'='*60}")
    
    # Check optimization output for performance data
    if kernel_name in optimization_results:
        opt_result = optimization_results[kernel_name]
        
        # Read output from saved file
        output_file = os.path.join(opt_result['results_dir'], "optimization_output.txt")
        error_file = os.path.join(opt_result['results_dir'], "optimization_errors.txt")
        
        output = ""
        if os.path.exists(output_file):
            with open(output_file, 'r') as f:
                output = f.read()
        else:
            output = opt_result['output']
        
        # Look for generated code files
        iter_files = glob.glob(os.path.join(opt_result['results_dir'], "iter_*.hip"))
        
        comp_entry = comparison_map.get(kernel_name)
        passed_compile = comp_entry['baseline'] if comp_entry else False
        all_matches = all(item['match'] for item in comp_entry['optimized']) if comp_entry else False
        passed_exec = passed_compile and all_matches
        
        baseline_perf = None
        optimized_perf = None
        best_iter_name = None
        speedup = None
        
        if comp_entry:
            baseline_perf = comp_entry.get('baseline_perf')
            matched_iters = [item for item in comp_entry['optimized'] if item['match'] and item.get('perf') is not None]
            if matched_iters:
                best_iter = min(matched_iters, key=lambda x: x['perf'])
                optimized_perf = best_iter['perf']
                best_iter_name = best_iter['iteration']
        
        if baseline_perf is None or optimized_perf is None:
            perf_patterns = [
                r"Candidate (\d+) perf[\s:]+([0-9.]+)",
                r"took\s+([0-9.]+)\s*ms",
                r"time:\s*([0-9.]+)\s*ms",
                r"latency:\s*([0-9.]+)\s*ms",
                r"([0-9.]+)\s*milliseconds"
            ]
            perf_matches = []
            for pattern in perf_patterns:
                matches = re.findall(pattern, output, re.IGNORECASE)
                if matches:
                    perf_matches.extend(matches)
            perfs = []
            for match in perf_matches:
                try:
                    if isinstance(match, tuple):
                        perfs.append(float(match[-1]))
                    else:
                        perfs.append(float(match))
                except:
                    continue
            if perfs:
                if optimized_perf is None:
                    optimized_perf = min(perfs)
                if baseline_perf is None and len(perfs) > 1:
                    baseline_perf = max(perfs)
            if baseline_perf is None:
                baseline_patterns = [
                    r"baseline.*?([0-9.]+)\s*ms",
                    r"original.*?([0-9.]+)\s*ms",
                    r"naive.*?([0-9.]+)\s*ms"
                ]
                for pattern in baseline_patterns:
                    baseline_match = re.search(pattern, output, re.IGNORECASE)
                    if baseline_match:
                        baseline_perf = float(baseline_match.group(1))
                        break
        
        if baseline_perf and optimized_perf and optimized_perf > 0:
            speedup = baseline_perf / optimized_perf
        
        # Display results
        status = "✓ PASS" if (opt_result['completed'] and passed_exec) else ("⚠ PARTIAL" if opt_result['completed'] else "✗ FAIL")
        print(f"  Status: {status}")
        print(f"  Compilation: {'✓ PASSED' if passed_compile else '✗ FAILED'}")
        print(f"  Execution: {'✓ PASSED' if passed_exec else '✗ FAILED'}")
        if not comp_entry:
            print("  (Run Step 7 to generate baseline vs optimized comparison data)")
        print(f"  Results Directory: {opt_result['results_dir']}")
        print(f"  Generated Files: {len(iter_files)} iteration file(s)")
        
        if best_iter_name:
            print(f"  Best Optimized Iteration: {best_iter_name}")
        
        if baseline_perf:
            print(f"  Baseline Performance: {baseline_perf:.4f} ms")
        if optimized_perf:
            print(f"  Optimized Performance: {optimized_perf:.4f} ms")
        if speedup:
            print(f"  Speedup: {speedup:.2f}x")
            improvement = (speedup - 1) * 100
            print(f"  Improvement: {improvement:.2f}%")
            print(f"  Improvement vs Baseline: {'✓ FASTER' if speedup > 1.0 else '✗ SLOWER'}")
        
        if baseline_perf and optimized_perf:
            plot_data.append({'Kernel': kernel_name, 'Baseline': baseline_perf, 'Optimized': optimized_perf})
        
        # Add to results table
        results_table.append({
            'Kernel': kernel_name,
            'Status': 'PASS' if (opt_result['completed'] and passed_exec) else ('PARTIAL' if opt_result['completed'] else 'FAIL'),
            'Correctness': 'PASS' if (passed_compile and passed_exec) else 'FAIL',
            'Baseline (ms)': f"{baseline_perf:.4f}" if baseline_perf else 'N/A',
            'Optimized (ms)': f"{optimized_perf:.4f}" if optimized_perf else 'N/A',
            'Speedup': f"{speedup:.2f}x" if speedup else 'N/A',
            'Better': '✓' if (speedup and speedup > 1.0) else '✗'
        })
    else:
        print(f"  No results found")
        results_table.append({
            'Kernel': kernel_name,
            'Status': 'N/A',
            'Correctness': 'N/A',
            'Baseline (ms)': 'N/A',
            'Optimized (ms)': 'N/A',
            'Speedup': 'N/A',
            'Better': 'N/A'
        })

# Display summary table
print("\n" + "="*80)
print("FINAL SUMMARY TABLE")
print("="*80)
if results_table:
    df = pd.DataFrame(results_table)
if results_table:
    df = pd.DataFrame(results_table)
    table_text = df.to_string(index=False)
    print(table_text)
    
    # Summary statistics
    print("\n" + "="*80)
    print("SUMMARY STATISTICS")
    print("="*80)
    completed = sum(1 for r in results_table if r['Status'] in ('PASS', 'PARTIAL'))
    correct = sum(1 for r in results_table if r['Correctness'] == 'PASS')
    improved = sum(1 for r in results_table if r['Better'] == '✓')
    print(f"Total Kernels: {len(results_table)}")
    print(f"Optimization Completed: {completed}/{len(results_table)}")
    print(f"Correctness Passed: {correct}/{len(results_table)}")
    print(f"Performance Improved: {improved}/{len(results_table)}")
    
    for row in results_table:
        kernel_dir = os.path.join(workspace_dir, f"results_{row['Kernel']}")
        os.makedirs(kernel_dir, exist_ok=True)
        kernel_summary = os.path.join(kernel_dir, "summary.txt")
        with open(kernel_summary, "w") as kf:
            kf.write("="*60 + "\n")
            kf.write(f"Kernel Summary: {row['Kernel']}\n")
            kf.write("="*60 + "\n")
            for key in ['Status', 'Correctness', 'Baseline (ms)', 'Optimized (ms)', 'Speedup', 'Better']:
                kf.write(f"{key}: {row[key]}\n")
            kf.write("\nGlobal Stats:\n")
            kf.write(f"Optimization Completed: {completed}/{len(results_table)}\n")
            kf.write(f"Correctness Passed: {correct}/{len(results_table)}\n")
            kf.write(f"Performance Improved: {improved}/{len(results_table)}\n")
    
    if plot_data:
        plot_df = pd.DataFrame(plot_data)
        plt.figure(figsize=(8, 4))
        ax = plt.gca()
        x = range(len(plot_df))
        width = 0.35
        baseline_bars = plt.bar([i - width/2 for i in x], plot_df['Baseline'], width=width, label='Baseline')
        optimized_bars = plt.bar([i + width/2 for i in x], plot_df['Optimized'], width=width, label='Optimized')
        plt.xticks(list(x), plot_df['Kernel'])
        plt.ylabel('Latency (ms)')
        plt.title('Baseline vs Optimized Performance')
        plt.legend()
        plt.tight_layout()
        for idx, row in plot_df.iterrows():
            speed = row['Baseline'] / row['Optimized'] if row['Optimized'] else float('nan')
            x_pos = idx
            y_pos = max(row['Baseline'], row['Optimized']) * 1.02
            ax.text(x_pos, y_pos, f"{speed:.2f}x", ha='center', va='bottom', fontsize=10, color='black')
        plt.show()
        
        for _, row in plot_df.iterrows():
            kernel_results_dir = os.path.join(workspace_dir, f"results_{row['Kernel']}")
            os.makedirs(kernel_results_dir, exist_ok=True)
            plt.figure(figsize=(8, 12))
            bars = plt.bar([0, 1], [row['Baseline'], row['Optimized']], color=['#8888ff', '#ff8888'])
            speed = row['Baseline'] / row['Optimized'] if row['Optimized'] else float('nan')
            plt.xticks([0, 1], ['Baseline', 'Optimized'])
            plt.ylabel('Latency (ms)')
            plt.title(f"{row['Kernel']} Performance\nSpeedup: {speed:.2f}x")
            plt.tight_layout()
            plot_path = os.path.join(kernel_results_dir, "baseline_vs_optimized.png")
            plt.savefig(plot_path, dpi=150)
            plt.close()
else:
    print("No results to display")